# Gnomonic Expansion and Binomial Matrices
Peter Luschny, August 2026

### A first look

A 'gnomonic expansion' takes a stream of numbers and turns it into a growing rectangular table — a bit like building Pascal's Triangle row by row, except here each new number wraps an L-shaped border (a "gnomon", the ancient Greek term for the piece you add to a square to make it one size bigger) around the existing table, adding both a new row and a new column.

One side of that new border (a new row) is built by *adding up* the recent values (running sums), and the other side (a new column) is built by *subtracting* them (running differences). So a single flat sequence gets unfolded into a square matrix where sums and differences of the original numbers sit next to each other.

Illustrating the algorithm: Assume a 0-based sequence a = [1, 1, 2, 5, ...] and the first three steps already finished that led to the 3×3 matrix. Now, compute a(3) = 5 and place it at the lower end of the diagonal. Add a new row starting from there, adding the entry on the left in the row above and repeat this until reaching the first column: 5 + 2 -> 7 + 3 -> 10 + 5 -> 15. Next, add a new column, starting again at the new diagonal term, and subtract the term in the previous row to its left; repeat this up to the first row: 5 - 2 -> 3 - 1 -> 2 - 1 = 1.

       1   0   1  |  1
       2   1   1  |  2
       5   3   2  |  3
       - - - - - - - -
      15  10   7  |  5

### The formal definition

We associate a matrix $A(n, k), \, (n \ge 0, k \ge 0), $ with a 0-based sequence $ a(0), a(1), \ldots ,$ defined by 
$$ A(n, k) = \sum_{m=0}^d \binom{d}{m} \, s^{d - m}  a(p + m),$$
where $d = |n - k|$, $p = \min(n, k)$, and $s = 1$ if $k \le n$, otherwise $-1$. 

We call this matrix the *binomial matrix* of $a$. We refer to the mapping $ a \mapsto A $ itself as the *gnomonic expansion* of $a$. The name describes the algorithmic, structural growth: each time a new term is appended to $a$, a new row is added below and a new column to the right of the existing matrix, transforming an $ n \times n $ matrix into an $ (n+1) \times (n+1) $ matrix. In this way, the given sequence $a$ is embedded as the main diagonal in the matrix, while the top row becomes the *inverse binomial transform* of $a$ and the left column becomes the *binomial transform* of $a$. Only simple arithmetic is used:  iterated sums and forward differences.

Given its particular arithmetic and algorithmic simplicity, the gnomonic expansion of a sequence is one of the fundamental methods for investigating the structure of a sequence of numbers. Combined with the concept of iterators, as built into computer languages ​​such as Python, it can be implemented very efficiently, as we show below.

### OEIS

A399000 is the main reference with an *alternative* implementation, also for Maple and Mathematica.

* A398987 Lucas numbers
* A398988 Powers n^n
* A398989 Sets of lists 
* A398990 Partition numbers 
* A398991 Fubini numbers 
* A398992 Central binomial coefficients
* A398993 Bell numbers
* A398994 (big) Schröder numbers
* A398995 Pell numbers
* A398996 Motzkin numbers
* A398997 Number of involutions
* A398998 Euler numbers
* A398999 Factorial numbers
* A399000 Catalan numbers
* A399001 Fibonacci numbers
and
* A398133/A398134 Bernoulli numbers

## A Python class for gnomonic expansion

In [1]:
from collections.abc import Iterator
from fractions import Fraction as frac
type Seq = list[int | frac]
type Matrix = list[Seq]
type SeqIterator = Iterator[int | frac]

class GnomonicExpansion:
    def __init__(self, seq: SeqIterator, dim: int | None = None) -> None:
        self.seq_source = seq
        self.matrix: Matrix = []
        if dim is not None and dim > 0:
            self.grow(dim)
        else:
            # dim = None: In this case the matrix grows until the given iterator 
            # stops (seq.next() raises StopIteration). The constant 99 is merely
            # a safety measure and was chosen arbitrarily.
            self.grow(99) 

    def grow(self, steps: int = 1) -> None:

        for _ in range(steps):
            try:
                r = next(self.seq_source)
            except StopIteration:
                break

            n = len(self.matrix)

            if n == 0:
                self.matrix.append([r])
                continue

            # Build the new lower row from the previous lower row slice and r.
            lower_new = self.matrix[-1][:] + [r]
            for k in range(n, 0, -1):
                lower_new[k - 1] += lower_new[k]

            # Build the new upper column from the previous upper column slice and r.
            prev_upper = [self.matrix[i][n - 1] for i in range(n)]
            upper_new = [0] * n + [r]
            for i in range(n - 1, -1, -1):
                upper_new[i] = upper_new[i + 1] - prev_upper[i]

            # Append the new upper values to existing rows.
            for i in range(n):
                self.matrix[i].append(upper_new[i])

            # Append the new lower row.
            self.matrix.append(lower_new)

    def get_matrix(self) -> Matrix:
        """Returns the current state of the binomial matrix."""
        return self.matrix

    def print_matrix(self, fraction: bool = False) -> None:
     
        matrix = self.matrix
        n = len(matrix)

        for row in matrix:
            print('[', ", ".join(str(v) for v in row), ']')
        print()

        diags = [[matrix[d - i][i] for i in range(d + 1)] for d in range(n)]
        for dia in diags:
            print('[', ", ".join(str(v.numerator) for v in dia), ']')
        print()

        if fraction:
            for dia in diags:
                print('[', ", ".join(str(v.denominator) for v in dia), ']')

# Catalan numbers

In [2]:
def catalan_iterator() -> SeqIterator:
    c, n = 1, 0
    while True:
        yield c
        c = c * (4 * n + 2) // (n + 2)
        n += 1

In [3]:
cat = GnomonicExpansion(catalan_iterator(), 8)
cat.print_matrix()

[ 1, 0, 1, 1, 3, 6, 15, 36 ]
[ 2, 1, 1, 2, 4, 9, 21, 51 ]
[ 5, 3, 2, 3, 6, 13, 30, 72 ]
[ 15, 10, 7, 5, 9, 19, 43, 102 ]
[ 51, 36, 26, 19, 14, 28, 62, 145 ]
[ 188, 137, 101, 75, 56, 42, 90, 207 ]
[ 731, 543, 406, 305, 230, 174, 132, 297 ]
[ 2950, 2219, 1676, 1270, 965, 735, 561, 429 ]

[ 1 ]
[ 2, 0 ]
[ 5, 1, 1 ]
[ 15, 3, 1, 1 ]
[ 51, 10, 2, 2, 3 ]
[ 188, 36, 7, 3, 4, 6 ]
[ 731, 137, 26, 5, 6, 9, 15 ]
[ 2950, 543, 101, 19, 9, 13, 21, 36 ]



# Lucas numbers

In [4]:
def lucas_iterator() -> SeqIterator:
    a, b = 2, 1
    while True:
        yield a
        a, b = b, a + b

In [5]:
luc = GnomonicExpansion(lucas_iterator(), 8)
luc.print_matrix()

[ 2, -1, 3, -4, 7, -11, 18, -29 ]
[ 3, 1, 2, -1, 3, -4, 7, -11 ]
[ 7, 4, 3, 1, 2, -1, 3, -4 ]
[ 18, 11, 7, 4, 3, 1, 2, -1 ]
[ 47, 29, 18, 11, 7, 4, 3, 1 ]
[ 123, 76, 47, 29, 18, 11, 7, 4 ]
[ 322, 199, 123, 76, 47, 29, 18, 11 ]
[ 843, 521, 322, 199, 123, 76, 47, 29 ]

[ 2 ]
[ 3, -1 ]
[ 7, 1, 3 ]
[ 18, 4, 2, -4 ]
[ 47, 11, 3, -1, 7 ]
[ 123, 29, 7, 1, 3, -11 ]
[ 322, 76, 18, 4, 2, -4, 18 ]
[ 843, 199, 47, 11, 3, -1, 7, -29 ]



# Euler numbers

In [ ]:
from math import comb

def euler_iterator() -> SeqIterator:
    seq = [1]
    yield 1
    n = 1
    sign = -1
    while True:
        yield 0  # yield 0 for odd indices 
        val = sum(comb(2 * n, 2 * k) * seq[n - k] * (1 if k % 2 == 1 else -1)
              for k in range(1, n + 1))
        seq.append(val)
        yield sign * val
        sign = -sign
        n += 1

In [7]:
eul = GnomonicExpansion(euler_iterator(), 9)
eul.print_matrix()

[ 1, -1, 0, 2, 0, -16, 0, 272, 0 ]
[ 1, 0, -1, 2, 2, -16, -16, 272, 272 ]
[ 0, -1, -1, 1, 4, -14, -32, 256, 544 ]
[ -2, -2, -1, 0, 5, -10, -46, 224, 800 ]
[ 0, 2, 4, 5, 5, -5, -56, 178, 1024 ]
[ 16, 16, 14, 10, 5, 0, -61, 122, 1202 ]
[ 0, -16, -32, -46, -56, -61, -61, 61, 1324 ]
[ -272, -272, -256, -224, -178, -122, -61, 0, 1385 ]
[ 0, 272, 544, 800, 1024, 1202, 1324, 1385, 1385 ]

[ 1 ]
[ 1, -1 ]
[ 0, 0, 0 ]
[ -2, -1, -1, 2 ]
[ 0, -2, -1, 2, 0 ]
[ 16, 2, -1, 1, 2, -16 ]
[ 0, 16, 4, 0, 4, -16, 0 ]
[ -272, -16, 14, 5, 5, -14, -16, 272 ]
[ 0, -272, -32, 10, 5, -10, -32, 272, 0 ]



# Motzkin numbers

In [2]:
def motzkin_iterator() -> SeqIterator:
    a, b = 1, 1
    yield a
    yield b
    n = 2
    while True:
        m = ((2 * n + 1) * b + (3 * n - 3) * a) // (n + 2)
        yield m
        a, b, n = b, m, n + 1

In [3]:
mot = GnomonicExpansion(motzkin_iterator(), 9)
mot.print_matrix()

[ 1, 0, 1, 0, 2, 0, 5, 0, 14 ]
[ 2, 1, 1, 1, 2, 2, 5, 5, 14 ]
[ 5, 3, 2, 2, 3, 4, 7, 10, 19 ]
[ 14, 9, 6, 4, 5, 7, 11, 17, 29 ]
[ 42, 28, 19, 13, 9, 12, 18, 28, 46 ]
[ 132, 90, 62, 43, 30, 21, 30, 46, 74 ]
[ 429, 297, 207, 145, 102, 72, 51, 76, 120 ]
[ 1430, 1001, 704, 497, 352, 250, 178, 127, 196 ]
[ 4862, 3432, 2431, 1727, 1230, 878, 628, 450, 323 ]

[ 1 ]
[ 2, 0 ]
[ 5, 1, 1 ]
[ 14, 3, 1, 0 ]
[ 42, 9, 2, 1, 2 ]
[ 132, 28, 6, 2, 2, 0 ]
[ 429, 90, 19, 4, 3, 2, 5 ]
[ 1430, 297, 62, 13, 5, 4, 5, 0 ]
[ 4862, 1001, 207, 43, 9, 7, 7, 5, 14 ]



# Pólya Trees
A000081, A034781, A375467

In [8]:
from math import isqrt

def polyatree_iterator()  -> SeqIterator:
    yield 0
    yield 1
    t = [0, 1]; t_append = t.append
    D = [0, 1]; D_append = D.append
    i = 2

    while True:
        total = sum(t[i - j] * D[j] for j in range(1, i))
        t_i = total // (i - 1)
        t_append(t_i)
        yield t_i

        d_i = 0
        for d in range(1, isqrt(i) + 1):
            if i % d == 0:
                d_i += d * t[d]
                if d * d != i:
                    d2 = i // d
                    d_i += d2 * t[d2]
        D_append(d_i)
        i += 1

In [9]:
pti = GnomonicExpansion(polyatree_iterator(), 9)
pti.print_matrix()

[ 0, 1, -1, 2, -2, 4, -5, 13, -25 ]
[ 1, 1, 0, 1, 0, 2, -1, 8, -12 ]
[ 3, 2, 1, 1, 1, 2, 1, 7, -4 ]
[ 8, 5, 3, 2, 2, 3, 3, 8, 3 ]
[ 22, 14, 9, 6, 4, 5, 6, 11, 11 ]
[ 64, 42, 28, 19, 13, 9, 11, 17, 22 ]
[ 195, 131, 89, 61, 42, 29, 20, 28, 39 ]
[ 615, 420, 289, 200, 139, 97, 68, 48, 67 ]
[ 1991, 1376, 956, 667, 467, 328, 231, 163, 115 ]

[ 0 ]
[ 1, 1 ]
[ 3, 1, -1 ]
[ 8, 2, 0, 2 ]
[ 22, 5, 1, 1, -2 ]
[ 64, 14, 3, 1, 0, 4 ]
[ 195, 42, 9, 2, 1, 2, -5 ]
[ 615, 131, 28, 6, 2, 2, -1, 13 ]
[ 1991, 420, 89, 19, 4, 3, 1, 8, -25 ]



# Sets of Lists

In [10]:
def sets_of_lists_iterator() -> SeqIterator:
    b, a, n = 1, 1, 2
    yield b
    yield a

    while True:
        q = (2 * n - 1) * a - (n - 1) * (n - 2) * b
        b, a, n = a, q, n + 1
        yield q

In [11]:
sol = GnomonicExpansion(sets_of_lists_iterator(), 9)
sol.print_matrix()

[ 1, 0, 2, 6, 36, 240, 1920, 17640, 183120 ]
[ 2, 1, 2, 8, 42, 276, 2160, 19560, 200760 ]
[ 6, 4, 3, 10, 50, 318, 2436, 21720, 220320 ]
[ 26, 20, 16, 13, 60, 368, 2754, 24156, 242040 ]
[ 148, 122, 102, 86, 73, 428, 3122, 26910, 266196 ]
[ 1032, 884, 762, 660, 574, 501, 3550, 30032, 293106 ]
[ 8464, 7432, 6548, 5786, 5126, 4552, 4051, 33582, 323138 ]
[ 79592, 71128, 63696, 57148, 51362, 46236, 41684, 37633, 356720 ]
[ 842832, 763240, 692112, 628416, 571268, 519906, 473670, 431986, 394353 ]

[ 1 ]
[ 2, 0 ]
[ 6, 1, 2 ]
[ 26, 4, 2, 6 ]
[ 148, 20, 3, 8, 36 ]
[ 1032, 122, 16, 10, 42, 240 ]
[ 8464, 884, 102, 13, 50, 276, 1920 ]
[ 79592, 7432, 762, 86, 60, 318, 2160, 17640 ]
[ 842832, 71128, 6548, 660, 73, 368, 2436, 19560, 183120 ]



## LCM -- Least common multiple of \{1, 2, ..., n\} 

In [12]:
def LCM_iterator(lng: int) -> SeqIterator:
    if lng <= 0: 
        print("This iterator requires a positive run length.")
        return

    lambd = [1] * lng
    lcm = [1] * lng
    isp = [True] * lng

    for p in range(2, lng):
        if isp[p]:
            for i in range(p * p, lng, p):
                isp[i] = False
            k = p
            while k < lng:
                lambd[k] = p
                k *= p
        lcm[p] = lcm[p - 1] * lambd[p]

    yield from lcm

In [13]:
gen = LCM_iterator(8)
lcm = GnomonicExpansion(gen)
lcm.print_matrix()

[ 1, 0, 1, 2, -3, 44, -215, 1014 ]
[ 2, 1, 1, 3, -1, 41, -171, 799 ]
[ 5, 3, 2, 4, 2, 40, -130, 628 ]
[ 16, 11, 8, 6, 6, 42, -90, 498 ]
[ 53, 37, 26, 18, 12, 48, -48, 408 ]
[ 206, 153, 116, 90, 72, 60, 0, 360 ]
[ 757, 551, 398, 282, 192, 120, 60, 360 ]
[ 2780, 2023, 1472, 1074, 792, 600, 480, 420 ]

[ 1 ]
[ 2, 0 ]
[ 5, 1, 1 ]
[ 16, 3, 1, 2 ]
[ 53, 11, 2, 3, -3 ]
[ 206, 37, 8, 4, -1, 44 ]
[ 757, 153, 26, 6, 2, 41, -215 ]
[ 2780, 551, 116, 18, 6, 40, -171, 1014 ]



# Bernoulli Matrix

In [14]:
def bernoulli_seidel() -> SeqIterator:
    """Generates Bernoulli numbers with B_1 = 1/2."""
    yield frac(1)     # B_0 = 1
    yield frac(1, 2)  # B_1 = 1/2

    ZERO = frac(0)  # odd-indexed Bernoulli number past B_1 vanish
    row = [1]       # B_0
    p2 = 8          # 2^(2+1) = 8
    m = 1           # row length

    while True:
        f = frac(row[-1], p2 - 2)
        yield -f if m % 2 == 0 else f  # yield B_n 
        yield ZERO  # yield B_{n+1} (always 0)
        row.append(0)
        p2 <<= 2
        m += 1
        for k in range(m - 2, -1, -1): row[k] += row[k + 1]
        for k in range(1, m): row[k] += row[k - 1]

In [15]:
ber = GnomonicExpansion(bernoulli_seidel(), 7)
ber.print_matrix(fraction=True)

[ 1, -1/2, 1/6, 0, -1/30, 0, 1/42 ]
[ 3/2, 1/2, -1/3, 1/6, -1/30, -1/30, 1/42 ]
[ 13/6, 2/3, 1/6, -1/6, 2/15, -1/15, -1/105 ]
[ 3, 5/6, 1/6, 0, -1/30, 1/15, -8/105 ]
[ 119/30, 29/30, 2/15, -1/30, -1/30, 1/30, -1/105 ]
[ 5, 31/30, 1/15, -1/15, -1/30, 0, 1/42 ]
[ 253/42, 43/42, -1/105, -8/105, -1/105, 1/42, 1/42 ]

[ 1 ]
[ 3, -1 ]
[ 13, 1, 1 ]
[ 3, 2, -1, 0 ]
[ 119, 5, 1, 1, -1 ]
[ 5, 29, 1, -1, -1, 0 ]
[ 253, 31, 2, 0, 2, -1, 1 ]

[ 1 ]
[ 2, 2 ]
[ 6, 2, 6 ]
[ 1, 3, 3, 1 ]
[ 30, 6, 6, 6, 30 ]
[ 1, 30, 6, 6, 30, 1 ]
[ 42, 30, 15, 1, 15, 30, 42 ]
